# Step 4 — Final Fraud Risk Scoring Engine

Loads the finalized XGBoost model and risk configuration from Step 6. No training, threshold tuning, or test-set evaluation.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
from pathlib import Path


## Load final model and configuration

In [2]:
model_dir = Path("../models")
model_path = model_dir / "final_xgb_model.pkl"
config_path = model_dir / "final_risk_config.pkl"

if not model_path.exists():
    raise FileNotFoundError(f"Missing final model: {model_path.resolve()}. Run Step 6 final-model save first.")
if not config_path.exists():
    raise FileNotFoundError(f"Missing final config: {config_path.resolve()}. Run Step 6 final-config save first.")

xgb_model = joblib.load(model_path)
final_config = joblib.load(config_path)
FINAL_THRESHOLD = float(final_config["threshold"])
COST_FALSE_POSITIVE = float(final_config["false_positive_cost"])
COST_FALSE_NEGATIVE = float(final_config["false_negative_cost"])

print("Final XGBoost model loaded successfully.")
print(final_config)
print(f"Final threshold: {FINAL_THRESHOLD:.2f}")


Final XGBoost model loaded successfully.
{'model': 'XGBoost', 'threshold': 0.03, 'false_positive_cost': 100.0, 'false_negative_cost': 5000.0}
Final threshold: 0.03


## Load feature schema

In [3]:
data_path = Path("../data/creditcard.csv")
df = pd.read_csv(data_path)
feature_columns = df.drop(columns=["Class"]).columns.tolist()
print("Number of features:", len(feature_columns))
print(feature_columns)
if hasattr(xgb_model, "n_features_in_") and xgb_model.n_features_in_ != len(feature_columns):
    raise ValueError(f"Feature mismatch: model expects {xgb_model.n_features_in_}, data has {len(feature_columns)}")


Number of features: 30
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']


In [4]:
explainer = shap.TreeExplainer(xgb_model)
print("SHAP TreeExplainer created successfully.")


SHAP TreeExplainer created successfully.


## Final transaction scoring engine

In [5]:
def score_transaction(transaction):
    if not isinstance(transaction, pd.DataFrame):
        raise TypeError("Transaction must be a pandas DataFrame.")
    if transaction.shape[0] != 1:
        raise ValueError("Transaction DataFrame must contain exactly one row.")

    missing_columns = [c for c in feature_columns if c not in transaction.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    transaction = transaction[feature_columns].copy()

    if transaction.isnull().any().any():
        raise ValueError("Transaction contains missing values.")
    if not all(pd.api.types.is_numeric_dtype(transaction[c]) for c in feature_columns):
        raise TypeError("All transaction features must be numeric.")

    fraud_probability = float(xgb_model.predict_proba(transaction)[0, 1])
    is_fraud = fraud_probability >= FINAL_THRESHOLD
    risk_level = "HIGH" if is_fraud else "LOW"

    shap_values = explainer.shap_values(transaction)
    shap_array = np.asarray(shap_values)
    if isinstance(shap_values, list):
        shap_row = np.asarray(shap_values[0]).reshape(-1)
    elif shap_array.ndim == 3:
        shap_row = shap_array[0, :, -1]
    else:
        shap_row = shap_array.reshape(-1)

    local_shap = pd.DataFrame({
        "feature": feature_columns,
        "feature_value": transaction.iloc[0].values,
        "shap_value": shap_row
    })
    local_shap["abs_shap"] = local_shap["shap_value"].abs()
    top_features = local_shap.sort_values("abs_shap", ascending=False).head(5).reset_index(drop=True)

    return {
        "fraud_probability": fraud_probability,
        "fraud_percentage": fraud_probability * 100,
        "threshold": FINAL_THRESHOLD,
        "is_fraud": bool(is_fraud),
        "risk_level": risk_level,
        "top_features": top_features[["feature", "feature_value", "shap_value"]]
    }


In [6]:
def display_risk_result(result):
    print("=" * 55)
    print("FRAUD RISK ASSESSMENT")
    print("=" * 55)
    print(f"Fraud Probability : {result['fraud_percentage']:.2f}%")
    print(f"Decision Threshold: {result['threshold']:.2f}")
    print(f"Risk Level        : {result['risk_level']}")
    print(f"Fraud Flag        : {result['is_fraud']}")
    print("\nTop Contributing Features:")
    for _, row in result["top_features"].iterrows():
        direction = "toward fraud" if row["shap_value"] > 0 else "away from fraud"
        print(f"- {row['feature']}: {direction} (SHAP={row['shap_value']:.4f})")


## Test 1 — Legitimate transaction

In [7]:
legitimate_transaction = df[df["Class"] == 0].drop(columns=["Class"]).iloc[[0]]
legitimate_result = score_transaction(legitimate_transaction)
display_risk_result(legitimate_result)


FRAUD RISK ASSESSMENT
Fraud Probability : 0.00%
Decision Threshold: 0.03
Risk Level        : LOW
Fraud Flag        : False

Top Contributing Features:
- V14: away from fraud (SHAP=-2.4304)
- V3: away from fraud (SHAP=-2.0143)
- V11: away from fraud (SHAP=-1.4473)
- V10: away from fraud (SHAP=-0.9671)
- V20: away from fraud (SHAP=-0.8415)


## Test 2 — Fraudulent transaction

In [8]:
fraud_transaction = df[df["Class"] == 1].drop(columns=["Class"]).iloc[[0]]
fraud_result = score_transaction(fraud_transaction)
display_risk_result(fraud_result)


FRAUD RISK ASSESSMENT
Fraud Probability : 99.99%
Decision Threshold: 0.03
Risk Level        : HIGH
Fraud Flag        : True

Top Contributing Features:
- V14: toward fraud (SHAP=4.0144)
- V10: toward fraud (SHAP=1.5497)
- V12: toward fraud (SHAP=1.3433)
- V8: away from fraud (SHAP=-0.9995)
- V4: toward fraud (SHAP=0.9752)


## Failure test — missing column

In [9]:
bad_transaction = legitimate_transaction.drop(columns=["Amount"])
try:
    score_transaction(bad_transaction)
except Exception as e:
    print(type(e).__name__)
    print(e)


ValueError
Missing required columns: ['Amount']


## Failure test — wrong input type

In [10]:
try:
    score_transaction("hello")
except Exception as e:
    print(type(e).__name__)
    print(e)


TypeError
Transaction must be a pandas DataFrame.


## Final configuration check

In [11]:
print("Model:", final_config["model"])
print("Threshold:", FINAL_THRESHOLD)
print("False positive cost:", COST_FALSE_POSITIVE)
print("False negative cost:", COST_FALSE_NEGATIVE)


Model: XGBoost
Threshold: 0.03
False positive cost: 100.0
False negative cost: 5000.0
